In [4]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name
# This section connects to MongoDB using the reusable CRUD module.
# The CRUD module acts as the interface between the dashboard and the database,
# allowing queries to be executed without writing raw MongoDB commands in the UI layer.

username = "aacuser"
password = "Compscirules404!"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)
    
# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)


## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


### Helper function ###
# This function builds MongoDB queries dynamically based on the selected
# rescue type. Each query reflects Grazioso Salvare’s real world criteria
# for identifying dogs suitable for specific training roles.
#
# Using this abstraction keeps filtering logic clean, reusable, and separate
# from the UI layer, improving maintainability.

def get_filtered_df(filter_type):
    # Water Rescue criteria
    if filter_type == "water":
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Labrador Retriever Mix",
                    "Chesapeake Bay Retriever",
                    "Newfoundland"
                ]
            },
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }
    # Mountain / Wilderness Rescue criteria
    elif filter_type == "mountain":
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "German Shepherd",
                    "Alaskan Malamute",
                    "Old English Sheepdog",
                    "Siberian Husky",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }
    # Disaster / Tracking criteria
    elif filter_type == "disaster":
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Doberman Pinscher",
                    "German Shepherd",
                    "Golden Retriever",
                    "Bloodhound",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }
    # Reset (no filtering)
    else:
        query = {}

    # Execute query through CRUD module
    filtered_df = pd.DataFrame.from_records(db.read(query))
    if not filtered_df.empty and '_id' in filtered_df.columns:
        filtered_df.drop(columns=['_id'], inplace=True)

    # Clean dataframe
    return filtered_df

#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
image_filename = "Grazioso Salvare Logo.png" # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode()

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([
    html.Div([
        html.A(
            html.Img(
                src=f"data:image/png;base64,{encoded_image}",
                style={"height": "180px", "display": "block", "margin": "0 auto"}
            ),
            href="https://www.snhu.edu",
            target="_blank"
        ),
        html.H1(
            "Grazioso Salvare Dashboard",
            style={"textAlign": "center", "marginBottom": "5px"}
        ),
        html.P(
            "Created by Jhasmin Gambon",
            style={"textAlign": "center", "fontSize": "18px", "marginTop": "0"}
        )
    ]),
        
#FIXME Add in code for the interactive filtering options. For example, Radio buttons, drop down, checkboxes, etc.

    html.Hr(),
    html.Div([
        html.Label("Filter by Rescue Type:", style={"fontWeight": "bold"}),
        dcc.RadioItems(
            id="filter-type",
            options=[
                {"label": " Reset", "value": "reset"},
                {"label": " Water Rescue", "value": "water"},
                {"label": " Mountain or Wilderness Rescue", "value": "mountain"},
                {"label": " Disaster or Individual Tracking", "value": "disaster"},
            ],
            value="reset",
            inline=True,
            style={"marginBottom": "15px"}
        )
    ], style={"margin": "10px"}),

    html.Hr(),

    dash_table.DataTable(
        id="datatable-id",
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True}
            for i in df.columns
        ],
        data=df.to_dict("records"),
        row_selectable="single",
        selected_rows=[0],
        sort_action="native",
        filter_action="native",
        page_action="native",
        page_current=0,
        page_size=10,
        style_table={"overflowX": "auto"},
        style_cell={
            "textAlign": "left",
            "minWidth": "120px",
            "width": "120px",
            "maxWidth": "180px",
            "whiteSpace": "normal"
        }
    ),
    
#FIXME: Set up the features for your interactive data table to make it user-friendly for your client
#If you completed the Module Six Assignment, you can copy in the code you created here 

                    
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
        style={"display": "flex", "gap": "20px", "alignItems": "flex-start"},
        children=[
            html.Div(id="graph-id", style={"width": "50%"}),
            html.Div(id="map-id", style={"width": "50%"})
        ]
    )
])

#############################################
# Interaction Between Components / Controller
#############################################
# Updates the DataTable based on selected rescue filter
# This demonstrates client side interaction triggering
# backend MongoDB queries via the CRUD module.


    
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
## FIX ME Add code to filter interactive data table with MongoDB queries
    dff = get_filtered_df(filter_type)
    return dff.to_dict("records")

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    ###FIX ME ####
    # add code for chart of your choice (e.g. pie chart) #

    #return [
    #    dcc.Graph(            
    #        figure = px.pie(df, names='breed', title='Preferred Animals')
    #    )    
    #]
    if viewData is None or len(viewData) == 0:
        return [html.P("No data available for this filter.")]

    dff = pd.DataFrame.from_dict(viewData)

    fig = px.pie(
        dff,
        names="breed",
        title="Breed Distribution"
    )

    return [dcc.Graph(figure=fig)]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if not selected_columns:
        return []

    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None or len(viewData) == 0:
        return [
            dl.Map(
                style={"width": "100%", "height": "500px"},
                center=[30.75, -97.48],
                zoom=10,
                children=[dl.TileLayer(id="base-layer-id")]
            )
        ]
    
    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    if row >= len(dff):
        row = 0
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={"width": "100%", "height": "500px"}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(
                position=[dff.iloc[row]["location_lat"], dff.iloc[row]["location_long"]],
                children=[
                    dl.Tooltip(dff.iloc[row]["breed"]),
                    dl.Popup([
                        html.H1("Animal Name"),
                        html.P(str(dff.iloc[row]["name"]))
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Dash app running on https://crownurban-dietflood-3000.codio.io/proxy/8050/
